# Hedge Fund OS — Agent Decision Demo

Shows how agents evaluate a stock and produce structured decisions.

**Flow:** Regime Agent → Technical Agent → Risk Agent → PM Agent

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import json
from src.data.ingest import download_ohlcv, get_spy
from src.data.features import compute_all_features, compute_spy_features
print('Setup complete.')

In [ ]:
# Pick a stock to analyze
TICKER = 'NVDA'

spy_raw = get_spy(start='2023-01-01')
spy = compute_spy_features(spy_raw)

stock_raw = download_ohlcv(TICKER, start='2023-01-01')
stock = compute_all_features(stock_raw)

# Use the most recent trading day
latest = stock.iloc[-1]
spy_latest = spy.iloc[-1]

print(f'Analyzing {TICKER} on {stock.index[-1].date()}')
print(f'Close: ${latest["close"]:.2f}')
print(f'Trend valid: {latest.get("trend_valid", False)}')
print(f'In base: {latest.get("in_base", False)}')
print(f'Breakout: {latest.get("breakout", False)}')
print(f'Volume ratio: {latest.get("vol_ratio", 0):.2f}')

## Run Each Agent

In [ ]:
from src.agents.regime_agent import RegimeAgent
from src.agents.technical_agent import TechnicalAgent
from src.agents.risk_agent import RiskAgent
from src.agents.pm_agent import PMAgent

def print_agent_output(output):
    """Pretty print an agent's structured output."""
    print(f'\n{"="*50}')
    print(f'  Agent: {output.agent}')
    print(f'  Direction: {output.direction.value}')
    print(f'  Confidence: {output.confidence}')
    print(f'  Thesis: {output.thesis}')
    print(f'  Evidence:')
    for e in output.evidence:
        print(f'    • {e}')
    print(f'  Risks:')
    for r in output.risks:
        print(f'    ⚠ {r}')
    print(f'  Invalidation:')
    for i in output.invalidation:
        print(f'    ✗ {i}')
    print(f'  Recommendation: {output.recommendation.value}')
    print(f'{"="*50}')

In [ ]:
# Step 1: Regime Agent
print('STEP 1: REGIME AGENT')
regime_agent = RegimeAgent()
regime_output = regime_agent.run({'spy_row': spy_latest})
if regime_output:
    print_agent_output(regime_output)

In [ ]:
# Step 2: Technical Agent
print('STEP 2: TECHNICAL AGENT')
tech_agent = TechnicalAgent()
tech_output = tech_agent.run({'ticker': TICKER, 'row': latest})
if tech_output:
    print_agent_output(tech_output)

In [ ]:
# Step 3: Risk Agent
print('STEP 3: RISK AGENT')
risk_agent = RiskAgent()
risk_output = risk_agent.run({
    'ticker': TICKER,
    'current_drawdown': -0.03,  # Simulated portfolio state
    'open_positions': 4,
    'max_positions': 10,
    'current_exposure_pct': 0.40,
})
if risk_output:
    print_agent_output(risk_output)
    print(f'\n  VETO: {risk_output.metadata.get("veto", False)}')

In [ ]:
# Step 4: PM Agent — Final Decision
print('STEP 4: PM AGENT (FINAL DECISION)')
pm_agent = PMAgent()
agent_outputs = [o for o in [regime_output, tech_output, risk_output] if o is not None]

pm_output = pm_agent.run({
    'ticker': TICKER,
    'agent_outputs': agent_outputs,
})
if pm_output:
    print_agent_output(pm_output)
    print(f'\n  FINAL VERDICT: {pm_output.direction.value.upper()}')
    print(f'  Long votes: {pm_output.metadata.get("long_votes", 0)}/{len(agent_outputs)}')

## Full Decision Log (JSON)

In [ ]:
# Full auditable decision record
decision_record = {
    'ticker': TICKER,
    'date': str(stock.index[-1].date()),
    'agents': []
}

for output in agent_outputs + ([pm_output] if pm_output else []):
    decision_record['agents'].append({
        'agent': output.agent,
        'direction': output.direction.value,
        'confidence': output.confidence,
        'thesis': output.thesis,
        'evidence': output.evidence[:3],
        'risks': output.risks[:3],
    })

print(json.dumps(decision_record, indent=2))